# 00：原版與排除高頻詞的 ticker 字典

以 `00_build_approx_crsp_universe.py` 為原版，原樣保留證券名稱關鍵字、ticker 長度、純字母、GME 等篩選規則。原版仍是公開資料的近似 CRSP 字典，不能視為實際 SHRCD 分類；本次不修正既有分類規則。

輸出兩份排序、去重、全大寫的字典：
1. `approx_common_stock_tickers.txt`：原版邏輯。
2. `approx_common_stock_tickers_excluding_top5000.txt`：從第一份刪除與前 5,000 個高頻詞相同的 ticker。

**來源與假設：**使用 [Peter Norvig 的 count_1w.txt](https://www.norvig.com/ngrams/)，該頁說明資料衍生自 Google Web Trillion Word Corpus。依次數遞減取前 5,000 個詞；同頻率時保留來源檔順序。先取全部詞的前 5,000 名，再與 ticker 比較，不先排除短詞。比較時轉為大寫，僅刪除完整相同字串，不做 substring、詞幹或人工補刪。

每次下載的來源文字與 SHA-256 保存在同一份 `source_snapshot.json`，可設定快照路徑離線重跑。原版股票來源的 main 分支可能改動，因此與歷史字典的差異會另外報告。

輸出位於 `Data/process/00_approx_crsp_universe/<UTC時間>/`。本 notebook 不更新 `pipeline_runs.json`，避免未經選擇就切換 02 使用的字典。兩版的實際差異、被刪 ticker 與詞頻排名都保存在單一摘要。此階段沒有 Reddit 作者、thread 或日期，對應 diagnostics 明確標為 N/A。


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import csv, io, re, json, hashlib, sys
import urllib.request
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Notebook/00_build_approx_crsp_universe.py').is_file())
sys.path.insert(0, str(ROOT/'Notebook'))
from pipeline_io import new_run
BASE = 'https://raw.githubusercontent.com/asad70/us_symbol_list/main/'
FILES = {'NASDAQ':'nasdaqsym.csv','NYSE':'nysesym.csv','AMEX':'amexsym.csv'}
WORD_URL = 'https://www.norvig.com/ngrams/count_1w.txt'
SOURCE_SNAPSHOT = None  # 如需離線重跑，指定先前 source_snapshot.json 的完整路徑。
RUN_DIR = new_run(ROOT,'universe')
pointer = ROOT/'Data/process/pipeline_runs.json'
pointer_before = pointer.read_bytes() if pointer.exists() else None
baseline_path = None
if pointer_before:
    manifest_path = json.loads(pointer_before).get('universe_manifest')
    if manifest_path:
        manifest_path = Path(manifest_path)
        if not manifest_path.is_absolute(): manifest_path = ROOT/manifest_path
        info = json.loads(manifest_path.read_text(encoding='utf-8'))
        baseline_path = manifest_path.parent/info.get('dictionary_file','approx_common_stock_tickers.txt')
print('輸出目錄：',RUN_DIR)


輸出目錄： C:\Users\eagle\RedditStock\Data\process\00_approx_crsp_universe\2026-09-22_09-26-22_UTC


## 1. 載入原版分類規則
以下規則與原版一致；不在此步驟變更研究篩選。

In [2]:
# Strong indicators that the listed security itself is not ordinary common equity.
# These are intentionally conservative: better to keep an ambiguous operating company
# for initial Reddit matching than to delete a true common stock by mistake.
EXCLUDE_PATTERNS = [
    r"\bETF\b",
    r"EXCHANGE[- ]TRADED FUND",
    r"EXCHANGE[- ]TRADED NOTE",
    r"\bETN\b",
    r"\bWARRANTS?\b",
    r"\bWTS?\b(?:\s|$)",
    r"\bRIGHTS?\b",
    r"\bUNITS?\b",
    r"\bPREFERRED\b",
    r"\bPFD\b",
    r"\bPREFERENCE SHARES?\b",
    r"\bDEPOSITARY SHARES?\b.*PREFERRED",
    r"\bCLOSED[- ]END\b",
    r"\bMUTUAL FUND\b",
    r"\bBOND FUND\b",
    r"\bINDEX FUND\b",
    r"\bTRUST PREFERRED\b",
    r"\bNOTES?\b.*(?:DUE|MATUR)",
]

# CRSP SHRCD 10/11 excludes ADR/ADS-type foreign receipts.
ADR_PATTERNS = [
    r"\bADR\b",
    r"\bADS\b",
    r"AMERICAN DEPOSITARY",
    r"DEPOSITARY RECEIPT",
    r"DEPOSITORY RECEIPT",
]

# Name patterns that are very likely a REIT/security rather than ordinary industrial common equity.
# This cannot catch every REIT; flagging is retained in the output for manual review.
REIT_PATTERNS = [
    r"\bREIT\b",
    r"REAL ESTATE INVESTMENT TRUST",
]

# Ticker suffixes frequently used for non-common securities.
# Do NOT blindly exclude any ticker ending in W/U/R/P because normal companies can end in these letters.
# We only use these as a secondary flag when the security name is also suspicious.
SUSPICIOUS_SUFFIX = re.compile(r"(?:W|WS|WT|U|R|P)$", re.I)

def fetch(url: str) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": "academic-replication/1.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read().decode("utf-8-sig", errors="replace")

def matches_any(text: str, patterns: list[str]) -> bool:
    return any(re.search(p, text, flags=re.I) for p in patterns)

def normalize_ticker(x: str) -> str:
    return x.strip().upper()



## 2. 取得來源並建立原版字典
下載三份股票清單與高頻詞資料，或使用指定快照。來源筆數與證券分類結果分別列出。

In [3]:
urls = [BASE+fn for fn in FILES.values()] + [WORD_URL]
if SOURCE_SNAPSHOT is None:
    source_text = {url:fetch(url) for url in urls}
else:
    source_text = json.loads(Path(SOURCE_SNAPSHOT).read_text(encoding='utf-8'))['sources']
    assert all(url in source_text for url in urls)
hashes = {url:hashlib.sha256(text.encode('utf-8')).hexdigest() for url,text in source_text.items()}
(RUN_DIR/'source_snapshot.json').write_text(json.dumps({'sources':source_text,'sha256_utf8_decoded_text':hashes},ensure_ascii=False),encoding='utf-8')
def fetch(url):
    return source_text[url]
for exchange,fn in FILES.items():
    print({'exchange':exchange,'source_rows':sum(1 for _ in csv.reader(io.StringIO(source_text[BASE+fn]))),
           'date_range':'N/A','unique_authors':'N/A','unique_threads':'N/A'})
rows = []
for exchange, fn in FILES.items():
    text = fetch(BASE + fn)
    reader = csv.reader(io.StringIO(text))
    for rec in reader:
        if len(rec) < 2:
            continue
        company = rec[0].strip()
        ticker = normalize_ticker(rec[1])
        if not company or not ticker:
            continue

        # Paper: keep tickers with at least 3 letters.
        # Requiring alphabetic tickers is intentional for Reddit token matching.
        if len(ticker) < 3 or not ticker.isalpha():
            continue

        # Paper excludes GameStop in the main sample.
        if ticker == "GME":
            continue

        upper_name = company.upper()

        is_adr = matches_any(upper_name, ADR_PATTERNS)
        is_reit = matches_any(upper_name, REIT_PATTERNS)
        is_noncommon = matches_any(upper_name, EXCLUDE_PATTERNS)

        # Name + suffix combination catches many SPAC units/warrants/rights.
        suffix_suspicious = bool(SUSPICIOUS_SUFFIX.search(ticker))
        name_suspicious = matches_any(
            upper_name,
            [
                r"\bSPAC\b",
                r"ACQUISITION CORPORATION",
                r"ACQUISITION CORP",
                r"\bWARRANT",
                r"\bUNIT",
                r"\bRIGHT",
                r"\bPREFERRED",
            ],
        )

        keep = not (is_adr or is_reit or is_noncommon or (suffix_suspicious and name_suspicious))

        reasons = []
        if is_adr:
            reasons.append("ADR/ADS")
        if is_reit:
            reasons.append("REIT")
        if is_noncommon:
            reasons.append("non-common security keyword")
        if suffix_suspicious and name_suspicious:
            reasons.append("suspicious ticker suffix + security name")

        rows.append({
            "ticker": ticker,
            "company_name": company,
            "exchange": exchange,
            "delisted_flag": int("(delisted)" in company.lower()),
            "approx_common_stock": int(keep),
            "exclusion_reason": "; ".join(reasons),
            "source_file": fn,
        })

# A ticker can appear in more than one historical record/exchange.
# Keep all records in the detailed output, and also write a deduplicated keep-list.
rows.sort(key=lambda x: (x["ticker"], x["exchange"], x["company_name"]))


keep_tickers = sorted({r['ticker'] for r in rows if r['approx_common_stock']==1})
assert keep_tickers
display(pd.DataFrame(rows).groupby(['exchange','approx_common_stock']).size().rename('rows').reset_index())
print({'security_rows':len(rows),'unique_kept_tickers':len(keep_tickers),'date_range':'N/A','unique_authors':'N/A','unique_threads':'N/A'})


{'exchange': 'NASDAQ', 'source_rows': 8030, 'date_range': 'N/A', 'unique_authors': 'N/A', 'unique_threads': 'N/A'}
{'exchange': 'NYSE', 'source_rows': 3469, 'date_range': 'N/A', 'unique_authors': 'N/A', 'unique_threads': 'N/A'}
{'exchange': 'AMEX', 'source_rows': 792, 'date_range': 'N/A', 'unique_authors': 'N/A', 'unique_threads': 'N/A'}


,exchange,approx_common_stock,rows
0,AMEX,0,233
1,AMEX,1,494
2,NASDAQ,0,483
3,NASDAQ,1,6754
4,NYSE,0,88
5,NYSE,1,3006


{'security_rows': 11058, 'unique_kept_tickers': 9804, 'date_range': 'N/A', 'unique_authors': 'N/A', 'unique_threads': 'N/A'}


## 3. 建立前 5,000 詞與篩選字典
驗證詞頻為正整數、來源詞無重複；以穩定排序處理同頻率。完整列出被刪 ticker 及其排名。

In [4]:
word_records = []
for line_number,line in enumerate(source_text[WORD_URL].splitlines(),1):
    if not line.strip(): continue
    word,count = line.split('\t')
    count = int(count)
    assert word and count > 0, line_number
    word_records.append((word,count))
assert len(word_records)>=5000
assert len({w.upper() for w,c in word_records})==len(word_records)
ranked = sorted(word_records,key=lambda item:-item[1])
top5000 = ranked[:5000]
high_frequency = {word.upper():{'rank':rank,'word':word,'count':count}
                  for rank,(word,count) in enumerate(top5000,1)}
removed = [dict(ticker=t,**high_frequency[t]) for t in keep_tickers if t in high_frequency]
filtered = [t for t in keep_tickers if t not in high_frequency]
assert set(filtered).isdisjoint(high_frequency)
assert set(filtered) | {r['ticker'] for r in removed} == set(keep_tickers)
assert len(filtered)+len(removed)==len(keep_tickers)
print({'source_word_rows':len(word_records),'top_words':len(top5000),
       'original_tickers':len(keep_tickers),'removed_tickers':len(removed),'filtered_tickers':len(filtered),
       'date_range':'N/A','unique_authors':'N/A','unique_threads':'N/A'})
display(pd.DataFrame([{'rank':i,'word':w,'count':c} for i,(w,c) in enumerate(top5000[:10],1)]))
with pd.option_context('display.max_rows',None):
    display(pd.DataFrame(removed).sort_values('rank') if removed else pd.DataFrame())


{'source_word_rows': 333333, 'top_words': 5000, 'original_tickers': 9804, 'removed_tickers': 321, 'filtered_tickers': 9483, 'date_range': 'N/A', 'unique_authors': 'N/A', 'unique_threads': 'N/A'}


,rank,word,count
0,1,the,23135851162
1,2,of,13151942776
2,3,and,12997637966
3,4,to,12136980858
4,5,a,9081174698
5,6,in,8469404971
6,7,for,5933321709
7,8,is,4705743816
8,9,on,3750423199
9,10,that,3400031103


,ticker,rank,word,count
107,FOR,7,for,5933321709
319,YOU,15,you,2996181025
14,ARE,20,are,2393614870
9,ALL,25,all,2022459848
203,NEW,27,new,1551258643
198,MORE,28,more,1544771673
134,HOME,33,home,1276852170
130,HAS,40,has,1046319984
113,FREE,42,free,1014107316
212,ONE,45,one,993536631


## 4. 輸出、歷史版本比較與驗證
保留兩份 ticker 字典、原版分類明細、一份來源快照與一份摘要。02 的既有字典選擇維持不變。

In [5]:
original_file = RUN_DIR/'approx_common_stock_tickers.txt'
filtered_file = RUN_DIR/'approx_common_stock_tickers_excluding_top5000.txt'
for path,values in [(original_file,keep_tickers),(filtered_file,filtered)]:
    path.write_text('\n'.join(values)+'\n',encoding='utf-8')
    assert path.read_text(encoding='utf-8').splitlines()==values
with (RUN_DIR/'approx_common_stock_universe.csv').open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0]))
    writer.writeheader();writer.writerows(rows)
comparison = {'baseline':str(baseline_path) if baseline_path else None}
if baseline_path and baseline_path.is_file():
    old=set(baseline_path.read_text(encoding='utf-8-sig').splitlines())
    comparison.update(identical=set(keep_tickers)==old,added=sorted(set(keep_tickers)-old),removed=sorted(old-set(keep_tickers)))
summary={'status':'complete','run_directory':str(RUN_DIR),'source_urls':urls,'source_sha256_utf8_decoded_text':hashes,
         'original_program_sha256':hashlib.sha256((ROOT/'Notebook/00_build_approx_crsp_universe.py').read_bytes()).hexdigest(),
         'security_records':len(rows),'original_ticker_count':len(keep_tickers),'filtered_ticker_count':len(filtered),
         'removed_ticker_count':len(removed),'removed_tickers_with_word_rank':removed,
         'word_source_rows':len(word_records),'top_word_count':5000,'cutoff_word':top5000[-1],
         'original_dictionary_file':original_file.name,'filtered_dictionary_file':filtered_file.name,
         'baseline_comparison':comparison,'pipeline_registry_updated':False,
         'date_range':None,'unique_authors':None,'unique_threads':None}
(RUN_DIR/'run_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
assert (pointer.read_bytes() if pointer.exists() else None)==pointer_before
print('原版字典：',original_file)
print('排除前 5,000 高頻詞的字典：',filtered_file)
print('原版與歷史字典比較：',comparison)


原版字典： C:\Users\eagle\RedditStock\Data\process\00_approx_crsp_universe\2026-09-22_09-26-22_UTC\approx_common_stock_tickers.txt
排除前 5,000 高頻詞的字典： C:\Users\eagle\RedditStock\Data\process\00_approx_crsp_universe\2026-09-22_09-26-22_UTC\approx_common_stock_tickers_excluding_top5000.txt
原版與歷史字典比較： {'baseline': None}
